# Reranking on a Colab GPU

Runs `api/rerank.py` from the GitHub repository on a free Colab GPU: all of train at N = 100 takes minutes here,
against more than an hour per system on the laptop's CPU. Same code, same pinned model; the run's `.json` records
the git commit and the device (`cuda`), and the scores go to their own cache file (`..._cuda.tsv`).

1. Runtime > Change runtime type > **T4 GPU**.
2. Runtime > Run all.
3. The last cell downloads `rerank_results.zip`. Unzip it into the project folder (`licenta`): it holds the new
   run files (`results/runs/*-rr*`) and the score cache (`results/work/rerank_cache/`).

Push first: Colab clones what is on GitHub, not what is on the laptop.

In [ ]:
import subprocess
REPO = 'https://github.com/ariszaharia/boolean-search.git'
COMMIT = ''   # empty = the newest pushed commit on master; each run's .json records which commit ran
subprocess.run(['git', 'clone', '-q', REPO, 'repo'], check=True)
%cd repo
if COMMIT:
    subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
!git log --oneline -1

In [ ]:
# The laptop's transformers version, so that both places run the same code.
!pip install -q transformers==5.19.0
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run all again.'
print(torch.cuda.get_device_name(0))

In [ ]:
# Train only: rerank.py refuses targeted and test until the reranker is fixed in hypotheses.md.
SPLIT = 'train'
SYSTEMS = ['B1', 'B2', 'B3']
for system in SYSTEMS:
    for n in [100, 30]:   # N = 30 comes from the cache filled by N = 100
        !python api/rerank.py --run results/runs/{system}_{SPLIT}.tsv --split {SPLIT} --n {n}

In [ ]:
!zip -q -r rerank_results.zip results/runs/*-rr* results/work/rerank_cache
from google.colab import files
files.download('rerank_results.zip')